# OPD Antibiotic Prescribing — Clustering Analysis

| | |
|---|---|
| **Study** | Developing a Framework for Rational Prescription of Antibiotics in EMR |
| **Institution** | Lady Ridgeway Hospital for Children, Colombo |
| **Researcher** | Dr. Chinthaka Jayarathne, MD Health Informatics Batch 05, PGIM |
| **Method** | CLARA (Clustering Large Applications) with Gower distance |
| **Reference** | Kaufman & Rousseeuw (1990). Finding Groups in Data. Wiley. |
| **Input** | `opd_prepared.csv` (output of data preparation notebook) |

---

## Why CLARA?

Computing a Gower distance matrix on 653,926 rows is not feasible (~400GB RAM).  
CLARA (Clustering Large Applications) solves this by:

1. Drawing **multiple independent random subsamples** (5 × 10,000 encounters)
2. Running **k-medoids on each subsample** separately to find candidate medoids
3. Evaluating each candidate set against the **full dataset**
4. Keeping the medoids with the **lowest total distance** across all 5 runs
5. **Assigning all encounters** to the best medoids found

This is more robust than a single sample — the final solution is the best across 5 independent attempts, not one lucky draw.

Only encounters from prescribers with **≥ 500 total encounters** are included,  
removing low-volume artefacts (locums, data entry errors).

## Steps
1. Setup & Data Load
2. Feature Preparation
3. Prescriber Filter
4. CLARA — Find optimal k (k = 3 to 8)
5. Choose k and finalise cluster assignments
6. Cluster Profiles
7. Visualisations
8. Save outputs

---
## Step 1 — Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import time
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import gower
import kmedoids

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── CONFIGURE ────────────────────────────────────────────────────────────────
PREPARED_DATA    = r'D:/Academic/MD Research 2025/AMS_2026/opd_prepared.csv'
FIGURES_DIR      = r'D:/Academic/MD Research 2025/AMS_2026/Clustering_figures'
OUTPUT_DIR       = r'D:/Academic/MD Research 2025/AMS_2026'

MIN_PRESCRIBER_ENCOUNTERS = 500  # remove low-volume prescribers
CLARA_N_SUBSAMPLES        = 5    # number of independent subsamples
CLARA_SUBSAMPLE_SIZE      = 10000 # encounters per subsample
RANDOM_SEED               = 42
K_RANGE                   = range(3, 9)  # k = 3 to 8

os.makedirs(FIGURES_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(FIGURES_DIR, f'{name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved: {path}')

print('Setup complete.')
print(f'CLARA settings: {CLARA_N_SUBSAMPLES} subsamples x {CLARA_SUBSAMPLE_SIZE:,} encounters each')

In [ ]:
# Load prepared dataset
df = pd.read_csv(PREPARED_DATA, low_memory=False)
df['DateTimeOfVisit'] = pd.to_datetime(df['DateTimeOfVisit'], errors='coerce')

# Fix MySQL \N nulls
df.replace(r'\N', np.nan, inplace=True)

# Fix numeric columns
for col in ['age_months', 'PatientWeight', 'complaint_duration_days',
            'visit_month', 'visit_hour']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fix weight outliers
df.loc[df['PatientWeight'] <= 0,  'PatientWeight'] = np.nan
df.loc[df['PatientWeight'] > 150, 'PatientWeight'] = np.nan

# Add visit_date for workload calculation
df['visit_date'] = df['DateTimeOfVisit'].dt.date

# Add prescriber_hourly_load if not already present
if 'prescriber_hourly_load' not in df.columns:
    load = (
        df.groupby(['prescriber_pseudo_id', 'visit_date', 'visit_hour'])
        .size()
        .reset_index(name='prescriber_hourly_load')
    )
    df = df.merge(load,
                  on=['prescriber_pseudo_id', 'visit_date', 'visit_hour'],
                  how='left')
    print('prescriber_hourly_load calculated and added.')

print(f'Loaded: {len(df):,} encounters, {df.shape[1]} columns')

---
## Step 2 — Feature Preparation

In [ ]:
# ── Define clustering features ────────────────────────────────────────────────
#
# Decision rationale (from EDA):
#   INCLUDED:
#     age_months              - independent patient characteristic
#     num_distinct_drugs      - polypharmacy indicator
#     num_antibiotics         - core prescribing intensity
#     prescriber_hourly_load  - workload / time pressure (EDA confirmed dose-response)
#     has_broad_spectrum      - prescribing quality signal
#     has_watch_antibiotic    - AWaRe stewardship signal
#     has_fever               - symptom flag
#     has_respiratory         - symptom flag
#     complaint_category      - clinical presentation (categorical)
#     age_stratum             - richer than raw months for clustering (categorical)
#
#   EXCLUDED:
#     has_antibiotic          - redundant with num_antibiotics (r=0.93 in EDA)
#     Gender                  - no variation in AB rate (48.5% vs 49.2%)
#     visit_month             - flat antibiotic rate across months
#     visit_hour              - replaced by prescriber_hourly_load
#     complaint_duration_days - 81% missing
#     PatientWeight           - extreme values + correlated with age

NUMERICAL_FEATURES   = ['age_months', 'num_distinct_drugs',
                         'num_antibiotics', 'prescriber_hourly_load']
BINARY_FEATURES      = ['has_broad_spectrum', 'has_watch_antibiotic',
                         'has_fever', 'has_respiratory']
CATEGORICAL_FEATURES = ['complaint_category', 'age_stratum']

ALL_FEATURES = NUMERICAL_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

# Categorical mask for gower.gower_matrix()
CAT_MASK = [col in CATEGORICAL_FEATURES for col in ALL_FEATURES]

print('Clustering features:')
print(f'  Numerical   ({len(NUMERICAL_FEATURES)}): {NUMERICAL_FEATURES}')
print(f'  Binary      ({len(BINARY_FEATURES)}): {BINARY_FEATURES}')
print(f'  Categorical ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}')
print(f'  Total: {len(ALL_FEATURES)} features')

In [ ]:
# ── Handle missing values then drop any remaining ─────────────────────────────
df['age_months']         = df['age_months'].fillna(df['age_months'].median())
df['complaint_category'] = df['complaint_category'].fillna('Unclassified')

n_before  = len(df)
df_clean  = df.dropna(subset=ALL_FEATURES).copy()
n_dropped = n_before - len(df_clean)

print(f'Rows before cleaning : {n_before:,}')
print(f'Rows after cleaning  : {len(df_clean):,}')
print(f'Rows dropped         : {n_dropped:,} ({100*n_dropped/n_before:.1f}%)')

print('\nMissing values in clustering features after cleaning:')
for col in ALL_FEATURES:
    n_miss = df_clean[col].isna().sum()
    print(f'  {col:<30}: {n_miss}')

---
## Step 3 — Prescriber Filter

In [ ]:
# ── Filter to prescribers with >= 500 encounters ──────────────────────────────
prescriber_counts  = df_clean['prescriber_pseudo_id'].value_counts()
active_prescribers = prescriber_counts[prescriber_counts >= MIN_PRESCRIBER_ENCOUNTERS].index

df_filtered = df_clean[df_clean['prescriber_pseudo_id'].isin(active_prescribers)].copy()
df_filtered = df_filtered.reset_index(drop=True)

print(f'Prescriber filter: >= {MIN_PRESCRIBER_ENCOUNTERS} encounters')
print(f'  Prescribers before : {len(prescriber_counts)}')
print(f'  Prescribers after  : {len(active_prescribers)}')
print(f'  Encounters before  : {len(df_clean):,}')
print(f'  Encounters after   : {len(df_filtered):,}')
print(f'  Removed            : {len(df_clean)-len(df_filtered):,} '
      f'({100*(len(df_clean)-len(df_filtered))/len(df_clean):.1f}%)')
print(f'\nActive prescribers and their encounter counts:')
print(prescriber_counts[prescriber_counts >= MIN_PRESCRIBER_ENCOUNTERS].to_string())

---
## Step 4 — CLARA: Find Optimal k

For each k (3 to 8), CLARA runs 5 independent subsamples of 10,000 encounters each.  
For each subsample it finds candidate medoids using k-medoids with Gower distance.  
The best medoid set (lowest total distance to full dataset) is kept.  
Silhouette score is computed on the full dataset using the best medoids.

In [ ]:
# ── CLARA implementation ──────────────────────────────────────────────────────

def gower_distance_to_medoids(X_all, medoid_rows, cat_mask):
    """
    Compute Gower distance from every row in X_all to each medoid row.
    Done in batches to manage memory.
    Returns array of shape (len(X_all), n_medoids).
    """
    n_medoids  = len(medoid_rows)
    n_all      = len(X_all)
    BATCH      = 5000
    dist_matrix = np.zeros((n_all, n_medoids), dtype=np.float32)

    for start in range(0, n_all, BATCH):
        end   = min(start + BATCH, n_all)
        batch = X_all.iloc[start:end]
        combined = pd.concat([batch, medoid_rows], ignore_index=True)
        dist     = gower.gower_matrix(combined, cat_features=cat_mask)
        dist_matrix[start:end, :] = dist[:len(batch), len(batch):]

    return dist_matrix


def run_clara(X, k, n_subsamples, subsample_size, cat_mask, random_seed=42):
    """
    CLARA algorithm.
    Returns: best_medoid_rows, best_labels, best_total_dist, all_subsample_losses
    """
    rng = np.random.default_rng(random_seed)
    best_total_dist  = np.inf
    best_medoid_rows = None
    best_labels      = None
    subsample_losses = []

    for s in range(n_subsamples):
        # Draw random subsample
        idx      = rng.choice(len(X), size=subsample_size, replace=False)
        X_sub    = X.iloc[idx].reset_index(drop=True)

        # Gower distance matrix on subsample
        dist_sub = gower.gower_matrix(X_sub, cat_features=cat_mask)

        # k-medoids on subsample
        km       = kmedoids.fasterpam(dist_sub, k,
                                      random_state=random_seed + s,
                                      max_iter=300)
        medoid_rows = X_sub.iloc[km.medoids].reset_index(drop=True)

        # Evaluate these medoids on the full dataset
        dist_to_medoids = gower_distance_to_medoids(X, medoid_rows, cat_mask)
        labels          = np.argmin(dist_to_medoids, axis=1)
        total_dist      = dist_to_medoids[np.arange(len(X)), labels].sum()
        subsample_losses.append(total_dist)

        print(f'    Subsample {s+1}/{n_subsamples}: total_dist={total_dist:.2f}')

        if total_dist < best_total_dist:
            best_total_dist  = total_dist
            best_medoid_rows = medoid_rows
            best_labels      = labels
            best_subsample   = s + 1

    print(f'    Best subsample: {best_subsample} (total_dist={best_total_dist:.2f})')
    return best_medoid_rows, best_labels, best_total_dist, subsample_losses


print('CLARA functions defined.')
print(f'Will run {CLARA_N_SUBSAMPLES} subsamples x {CLARA_SUBSAMPLE_SIZE:,} encounters for each k.')

In [ ]:
# ── Run CLARA for k = 3 to 8 ─────────────────────────────────────────────────
# Expected time: 5-20 minutes per k depending on your computer.
# Total: 30-120 minutes for all 6 values of k.

X_full = df_filtered[ALL_FEATURES].copy()

clara_results = {}

for k in K_RANGE:
    print(f'\n{"="*55}')
    print(f'Running CLARA for k = {k}...')
    t0 = time.time()

    medoid_rows, labels, total_dist, sub_losses = run_clara(
        X_full, k,
        n_subsamples  = CLARA_N_SUBSAMPLES,
        subsample_size= CLARA_SUBSAMPLE_SIZE,
        cat_mask      = CAT_MASK,
        random_seed   = RANDOM_SEED
    )

    # Silhouette on a 5,000-row random sample (full dataset too large)
    sil_idx    = np.random.default_rng(RANDOM_SEED).choice(len(X_full),
                                       size=min(5000, len(X_full)), replace=False)
    X_sil      = X_full.iloc[sil_idx].reset_index(drop=True)
    labels_sil = labels[sil_idx]
    dist_sil   = gower.gower_matrix(X_sil, cat_features=CAT_MASK)
    sil_score  = silhouette_score(dist_sil, labels_sil, metric='precomputed')

    elapsed = time.time() - t0
    clara_results[k] = {
        'medoid_rows' : medoid_rows,
        'labels'      : labels,
        'total_dist'  : total_dist,
        'sub_losses'  : sub_losses,
        'silhouette'  : sil_score,
    }
    print(f'  k={k}: silhouette={sil_score:.4f}, '
          f'total_dist={total_dist:.2f} ({elapsed/60:.1f} min)')

print(f'\nAll k values complete.')

In [ ]:
# ── Silhouette score plot ─────────────────────────────────────────────────────
ks   = list(K_RANGE)
sils = [clara_results[k]['silhouette'] for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Silhouette
axes[0].plot(ks, sils, marker='o', color='#E05C5C', linewidth=2)
for k, s in zip(ks, sils):
    axes[0].annotate(f'{s:.4f}', (k, s),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=10)
axes[0].set_title('Silhouette Score by k\n(higher = better separated clusters)',
                  fontweight='bold')
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Silhouette score')
axes[0].set_xticks(ks)
axes[0].set_ylim(0, max(sils) * 1.3)

# Total distance (elbow)
total_dists = [clara_results[k]['total_dist'] for k in ks]
axes[1].plot(ks, total_dists, marker='o', color='#5B9BD5', linewidth=2)
for k, d in zip(ks, total_dists):
    axes[1].annotate(f'{d:.0f}', (k, d),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=9)
axes[1].set_title('Total Distance by k\n(look for elbow — flattening point)',
                  fontweight='bold')
axes[1].set_xlabel('k (number of clusters)')
axes[1].set_ylabel('Total distance (full dataset)')
axes[1].set_xticks(ks)

plt.tight_layout()
savefig('C01_k_selection')
plt.show()

best_k_auto = max(clara_results, key=lambda k: clara_results[k]['silhouette'])
print(f'Highest silhouette: k={best_k_auto} '
      f'({clara_results[best_k_auto]["silhouette"]:.4f})')
print('Review both plots. Choose k based on silhouette AND clinical interpretability.')

---
## Step 5 — Choose k and Finalise Assignments

In [ ]:
# ── SET YOUR CHOSEN k HERE ────────────────────────────────────────────────────
# Look at the silhouette plot and elbow plot above.
# Best k by silhouette is auto-detected above.
# You can override if a different k is more clinically meaningful.

BEST_K = best_k_auto   # <── change this number if needed

# Apply cluster labels to full filtered dataset
df_filtered['cluster'] = clara_results[BEST_K]['labels']

print(f'Selected k = {BEST_K}')
print(f'Silhouette score: {clara_results[BEST_K]["silhouette"]:.4f}')
print(f'Total distance  : {clara_results[BEST_K]["total_dist"]:.2f}')
print(f'\nCluster sizes:')
for c, n in df_filtered['cluster'].value_counts().sort_index().items():
    print(f'  Cluster {c}: {n:,} ({100*n/len(df_filtered):.1f}%)')

In [ ]:
# ── Subsample stability check ─────────────────────────────────────────────────
# Shows how consistent the 5 subsamples were for the chosen k.
# Similar total_dist values = stable solution.

losses = clara_results[BEST_K]['sub_losses']
print(f'Subsample total distances for k={BEST_K}:')
for i, l in enumerate(losses):
    marker = '  <-- best (selected)' if l == min(losses) else ''
    print(f'  Subsample {i+1}: {l:.2f}{marker}')
print(f'\nRange: {max(losses)-min(losses):.2f}')
print(f'CV   : {np.std(losses)/np.mean(losses)*100:.1f}%')
print('(Low CV = consistent solution across subsamples)')

---
## Step 6 — Cluster Profiles

In [ ]:
# ── Numerical and binary summary per cluster ──────────────────────────────────
profile_cols = [
    'age_months', 'num_distinct_drugs', 'num_antibiotics',
    'prescriber_hourly_load', 'has_antibiotic',
    'has_broad_spectrum', 'has_watch_antibiotic',
    'has_fever', 'has_respiratory'
]

profile = df_filtered.groupby('cluster')[profile_cols].mean().round(3)
profile['n']   = df_filtered.groupby('cluster').size()
profile['pct'] = (profile['n'] / len(df_filtered) * 100).round(1)

for col in ['has_antibiotic', 'has_broad_spectrum', 'has_watch_antibiotic',
            'has_fever', 'has_respiratory']:
    profile[col] = (profile[col] * 100).round(1)

profile = profile.rename(columns={
    'has_antibiotic':      'ab_rate_%',
    'has_broad_spectrum':  'broad_%',
    'has_watch_antibiotic':'watch_%',
    'has_fever':           'fever_%',
    'has_respiratory':     'resp_%',
})

print('CLUSTER PROFILES — numerical summary')
print('=' * 80)
print(profile.T.to_string())

In [ ]:
# ── Dominant complaint category per cluster ───────────────────────────────────
print('Dominant complaint categories per cluster:')
print('=' * 60)
for c in sorted(df_filtered['cluster'].unique()):
    sub = df_filtered[df_filtered['cluster'] == c]
    top = sub['complaint_category'].value_counts().head(4)
    print(f'\nCluster {c} (n={len(sub):,}, AB rate={sub["has_antibiotic"].mean()*100:.1f}%)')
    for cat, n in top.items():
        print(f'  {cat:<25}: {n:>6,} ({100*n/len(sub):.1f}%)')

In [ ]:
# ── Age stratum per cluster ───────────────────────────────────────────────────
print('Age stratum distribution per cluster:')
print('=' * 60)
age_order = ['neonate_0_27d','infant_1_2m','infant_3_5m','infant_6_11m',
             'toddler_12_23m','preschool_2_4y','child_5_11y',
             'adolescent_12_17y','adult_18plus']
for c in sorted(df_filtered['cluster'].unique()):
    sub = df_filtered[df_filtered['cluster'] == c]
    top = sub['age_stratum'].value_counts().head(3)
    print(f'\nCluster {c}')
    for stratum, n in top.items():
        print(f'  {stratum:<25}: {n:>6,} ({100*n/len(sub):.1f}%)')

In [ ]:
# ── Top antibiotics per cluster ───────────────────────────────────────────────
print('Top 5 antibiotics per cluster (antibiotic encounters only):')
print('=' * 60)
for c in sorted(df_filtered['cluster'].unique()):
    sub = df_filtered[
        (df_filtered['cluster'] == c) &
        (df_filtered['has_antibiotic'] == 1)
    ]
    if len(sub) == 0:
        print(f'Cluster {c}: no antibiotic encounters')
        continue
    top_ab = (
        sub['antibiotic_compounds']
        .dropna()
        .str.split('|')
        .explode()
        .str.strip()
        .value_counts()
        .head(5)
    )
    print(f'\nCluster {c}')
    for drug, n in top_ab.items():
        print(f'  {drug:<40}: {n:>5,}')

---
## Step 7 — Visualisations

In [ ]:
# ── Cluster size bar chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sizes  = df_filtered['cluster'].value_counts().sort_index()
colors = plt.cm.Set2(np.linspace(0, 1, len(sizes)))
bars   = ax.bar([f'Cluster {c}' for c in sizes.index],
                sizes.values, color=colors, edgecolor='white')
for bar, v in zip(bars, sizes.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 200,
            f'{v:,}\n({100*v/len(df_filtered):.1f}%)',
            ha='center', fontsize=9)
ax.set_title('Cluster Sizes', fontsize=13, fontweight='bold')
ax.set_ylabel('Encounters')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
savefig('C02_cluster_sizes')
plt.show()

In [ ]:
# ── Cluster profile heatmap ───────────────────────────────────────────────────
heatmap_cols = ['ab_rate_%', 'broad_%', 'watch_%', 'fever_%', 'resp_%',
                'age_months', 'num_distinct_drugs',
                'num_antibiotics', 'prescriber_hourly_load']

heatmap_data = profile[heatmap_cols].T

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heatmap_data.astype(float),
            annot=True, fmt='.1f',
            cmap='RdYlGn_r', linewidths=0.5,
            ax=ax, cbar_kws={'label': 'Value'})
ax.set_title(f'Cluster Profile Heatmap (CLARA, k={BEST_K})',
             fontsize=13, fontweight='bold')
ax.set_xticklabels([f'Cluster {c}' for c in profile.index])
plt.tight_layout()
savefig('C03_cluster_heatmap')
plt.show()

In [ ]:
# ── PCA visualisation ─────────────────────────────────────────────────────────
# PCA on a 10,000-row sample of numerical + binary features for 2D display
print('Running PCA for 2D visualisation...')

pca_idx    = np.random.default_rng(RANDOM_SEED).choice(
                 len(df_filtered), size=min(10000, len(df_filtered)), replace=False)
df_pca_sub = df_filtered.iloc[pca_idx].copy()

num_cols_pca = NUMERICAL_FEATURES + BINARY_FEATURES
X_pca_mat    = df_pca_sub[num_cols_pca].astype(float)
scaler       = StandardScaler()
X_scaled     = scaler.fit_transform(X_pca_mat)

pca          = PCA(n_components=2, random_state=RANDOM_SEED)
coords       = pca.fit_transform(X_scaled)
var_exp      = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(10, 7))
scatter_colors = plt.cm.Set2(np.linspace(0, 1, BEST_K))

for c in range(BEST_K):
    mask = df_pca_sub['cluster'].values == c
    ax.scatter(coords[mask, 0], coords[mask, 1],
               alpha=0.3, s=15,
               color=scatter_colors[c],
               label=f'Cluster {c} (n={mask.sum():,})')

ax.set_title(f'PCA — Cluster Visualisation (CLARA, k={BEST_K})\n'
             f'PC1: {var_exp[0]:.1f}%  PC2: {var_exp[1]:.1f}% variance explained',
             fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
savefig('C04_pca_clusters')
plt.show()

In [ ]:
# ── AB rate + complaint mix per cluster ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AB rate per cluster
ab_rates   = profile['ab_rate_%'].sort_values(ascending=False)
bar_colors = plt.cm.Set2(np.linspace(0, 1, len(ab_rates)))
bars = axes[0].bar([f'Cluster {c}' for c in ab_rates.index],
                   ab_rates.values, color=bar_colors, edgecolor='white')
for bar, v in zip(bars, ab_rates.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1, f'{v:.1f}%',
                 ha='center', fontsize=10)
axes[0].axhline(df_filtered['has_antibiotic'].mean()*100,
                color='grey', linestyle='--',
                label=f'Overall avg ({df_filtered["has_antibiotic"].mean()*100:.1f}%)')
axes[0].set_title('Antibiotic Rate per Cluster', fontweight='bold')
axes[0].set_ylabel('Antibiotic rate (%)')
axes[0].set_ylim(0, 110)
axes[0].legend()

# Complaint category stacked bar
cat_pivot     = (
    df_filtered.groupby(['cluster', 'complaint_category'])
    .size().unstack(fill_value=0)
)
cat_pivot_pct = cat_pivot.div(cat_pivot.sum(axis=1), axis=0) * 100
cat_pivot_pct.plot(kind='bar', stacked=True, ax=axes[1],
                   colormap='tab20', edgecolor='none')
axes[1].set_title('Complaint Category Mix per Cluster', fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('%')
axes[1].set_xticklabels([f'Cluster {c}' for c in cat_pivot_pct.index], rotation=0)
axes[1].legend(title='Category', bbox_to_anchor=(1.01, 1),
               loc='upper left', fontsize=7)

plt.tight_layout()
savefig('C05_cluster_ab_and_complaints')
plt.show()

---
## Step 8 — Save Outputs

In [ ]:
# ── Save clustered dataset ────────────────────────────────────────────────────
CLUSTERED_PATH = os.path.join(OUTPUT_DIR, 'opd_clustered.csv')
df_filtered.to_csv(CLUSTERED_PATH, index=False)
print(f'Saved: {CLUSTERED_PATH}')
print(f'  Rows: {len(df_filtered):,}, Columns: {df_filtered.shape[1]}')

# ── Save cluster profiles ─────────────────────────────────────────────────────
PROFILE_PATH = os.path.join(OUTPUT_DIR, 'cluster_profiles.csv')
profile.to_csv(PROFILE_PATH)
print(f'Saved: {PROFILE_PATH}')

# ── Final summary ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CLUSTERING COMPLETE')
print(f'{"="*60}')
print(f'  Method              : CLARA ({CLARA_N_SUBSAMPLES} x {CLARA_SUBSAMPLE_SIZE:,})')
print(f'  k selected          : {BEST_K}')
print(f'  Silhouette score    : {clara_results[BEST_K]["silhouette"]:.4f}')
print(f'  Prescribers included: {df_filtered["prescriber_pseudo_id"].nunique()}')
print(f'  Encounters clustered: {len(df_filtered):,}')
print(f'  Figures saved to    : {FIGURES_DIR}')
print(f'\nCluster summary:')
for c in sorted(df_filtered['cluster'].unique()):
    sub = df_filtered[df_filtered['cluster'] == c]
    ab  = sub['has_antibiotic'].mean() * 100
    dom = sub['complaint_category'].value_counts().index[0]
    load = sub['prescriber_hourly_load'].mean()
    print(f'  Cluster {c}: n={len(sub):,} ({100*len(sub)/len(df_filtered):.1f}%) '
          f'| AB={ab:.1f}% | load={load:.1f} | dominant={dom}')

print('\nNext step: Archetype naming and thesis write-up')